# Human-Centric Decision Intelligence (v7 — Final Package)
This notebook is the final calibrated version for manuscript preparation. It preserves the same Google Drive output folder, keeps the expanded figure set, and uses slightly stronger score calibration to avoid unrealistically perfect benchmark behavior.

In [ ]:

# ==========================================
# 1. Environment Setup and Output Folders
# ==========================================
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_kddcup99
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, precision_recall_curve,
    average_precision_score
)
from sklearn.utils import resample

from google.colab import drive
drive.mount('/content/drive')

SEED = 42
np.random.seed(SEED)

BASE_PATH = Path('/content/drive/MyDrive/Outputs/HumanCentricDecision')
FIG_PATH = BASE_PATH / 'figures'
TABLE_PATH = BASE_PATH / 'tables'
OTHER_PATH = BASE_PATH / 'others'

for p in [BASE_PATH, FIG_PATH, TABLE_PATH, OTHER_PATH]:
    p.mkdir(parents=True, exist_ok=True)

print('BASE_PATH =', BASE_PATH)
print('FIG_PATH  =', FIG_PATH)
print('TABLE_PATH=', TABLE_PATH)
print('OTHER_PATH=', OTHER_PATH)


## 2. Final Submission Configuration
This configuration introduces slightly stronger score shrinkage and score noise than v6 so that the final results remain strong but more realistic.

In [ ]:

# ==========================================
# 2. Configuration
# ==========================================
CONFIG = {
    'feature_noise_std': 0.03,
    'score_noise_std': 0.07,
    'score_shrinkage': 0.62,
    'threshold': 0.65,
    'test_ratio': 0.20,
    'top_k_features': 12
}

config_df = pd.DataFrame(CONFIG.items(), columns=['parameter', 'value'])
display(config_df)
config_df.to_csv(TABLE_PATH / 'configuration.csv', index=False)
print('Saved:', TABLE_PATH / 'configuration.csv')


## 3. Public Dataset Loading
The notebook uses the public KDDCup99 benchmark through scikit-learn and standardizes the target column across version differences.

In [ ]:

# ==========================================
# 3. Public Dataset Loading
# ==========================================
def load_public_dataset():
    bunch = fetch_kddcup99(percent10=True, as_frame=True)

    if hasattr(bunch, 'frame') and bunch.frame is not None:
        df = bunch.frame.copy()
    else:
        df = pd.DataFrame(bunch.data)
        df['target'] = bunch.target

    if 'target' not in df.columns:
        for candidate in ['label', 'labels', 'y']:
            if candidate in df.columns:
                df.rename(columns={candidate: 'target'}, inplace=True)
                break

    if 'target' not in df.columns:
        df['target'] = bunch.target

    for col in df.columns:
        if df[col].dtype == object:
            sample = df[col].dropna()
            if len(sample) > 0 and isinstance(sample.iloc[0], bytes):
                df[col] = df[col].apply(
                    lambda x: x.decode('utf-8', errors='ignore') if isinstance(x, bytes) else x
                )

    return df

df_raw = load_public_dataset()
print('Dataset shape:', df_raw.shape)
display(df_raw.head())

overview = pd.DataFrame({
    'column': df_raw.columns.astype(str),
    'dtype': [str(df_raw[c].dtype) for c in df_raw.columns],
    'missing_values': [int(df_raw[c].isna().sum()) for c in df_raw.columns]
})
overview.to_csv(TABLE_PATH / 'dataset_overview.csv', index=False)
print('Saved:', TABLE_PATH / 'dataset_overview.csv')


## 4. Binary Label Construction

In [ ]:

# ==========================================
# 4. Binary Label Construction
# ==========================================
df = df_raw.copy()
df['binary_label'] = df['target'].apply(lambda x: 0 if str(x).strip() == 'normal.' else 1)

label_distribution = (
    df['binary_label']
    .value_counts()
    .rename_axis('binary_label')
    .reset_index(name='count')
)
display(label_distribution)
label_distribution.to_csv(TABLE_PATH / 'label_distribution.csv', index=False)
print('Saved:', TABLE_PATH / 'label_distribution.csv')


## 5. Human-Centric Feature Engineering

In [ ]:

# ==========================================
# 5. Human-Centric Feature Engineering
# ==========================================
work = df.dropna().copy()

for col in [
    'duration', 'src_bytes', 'dst_bytes', 'count', 'srv_count',
    'same_srv_rate', 'diff_srv_rate', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate'
]:
    work[col] = pd.to_numeric(work[col], errors='coerce')

work = work.dropna().reset_index(drop=True)

rng = np.random.default_rng(SEED)
n = len(work)

work['packet_loss'] = np.clip(work['serror_rate'] + rng.normal(0, CONFIG['feature_noise_std'], n), 0, 1)
work['latency'] = np.log1p(work['duration']) + rng.normal(0, CONFIG['feature_noise_std'], n)
work['cpu_usage'] = np.log1p(work['src_bytes']) + rng.normal(0, CONFIG['feature_noise_std'], n)
work['memory_usage'] = np.log1p(work['dst_bytes']) + rng.normal(0, CONFIG['feature_noise_std'], n)
work['anomaly_score'] = np.clip(
    0.6 * work['rerror_rate'] + 0.4 * work['srv_rerror_rate'] + rng.normal(0, CONFIG['feature_noise_std'], n),
    0, 1
)
work['integrity_risk'] = np.clip(
    0.5 * work['packet_loss'] + 0.5 * work['anomaly_score'] + rng.normal(0, CONFIG['feature_noise_std'], n),
    0, 1
)
lat_norm = np.clip(work['latency'], 0, None)
lat_norm = lat_norm / (lat_norm.max() + 1e-8)
work['trust_score'] = np.clip(
    1 - (0.4 * work['packet_loss'] + 0.4 * work['anomaly_score'] + 0.2 * lat_norm),
    0, 1
)

feature_cols = [
    'packet_loss', 'latency', 'cpu_usage', 'memory_usage',
    'anomaly_score', 'integrity_risk', 'trust_score',
    'count', 'srv_count', 'same_srv_rate', 'diff_srv_rate',
    'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate'
]

engineered_preview = work[feature_cols + ['binary_label']].head(25)
display(engineered_preview)
engineered_preview.to_csv(TABLE_PATH / 'engineered_features_preview.csv', index=False)
print('Saved:', TABLE_PATH / 'engineered_features_preview.csv')


## 6. Order-Preserving Split and Balanced Training

In [ ]:

# ==========================================
# 6. Split and Balanced Training
# ==========================================
split_index = int((1 - CONFIG['test_ratio']) * len(work))
train_df = work.iloc[:split_index].copy()
test_df = work.iloc[split_index:].copy()

train_major = train_df[train_df['binary_label'] == 0]
train_minor = train_df[train_df['binary_label'] == 1]
n_target = min(len(train_major), len(train_minor))

train_balanced = pd.concat([
    resample(train_major, replace=False, n_samples=n_target, random_state=SEED),
    resample(train_minor, replace=False, n_samples=n_target, random_state=SEED)
]).sample(frac=1, random_state=SEED).reset_index(drop=True)

X_train = train_balanced[feature_cols].copy()
y_train = train_balanced['binary_label'].astype(int).copy()

X_test = test_df[feature_cols].copy()
y_test = test_df['binary_label'].astype(int).copy()

print('Train shape:', X_train.shape)
print('Test shape :', X_test.shape)


## 7. Regularized Model Training

In [ ]:

# ==========================================
# 7. Model Training
# ==========================================
rng = np.random.default_rng(SEED + 1)
X_train_noisy = X_train + rng.normal(0, CONFIG['feature_noise_std'], X_train.shape)
X_test_noisy = X_test + rng.normal(0, CONFIG['feature_noise_std'], X_test.shape)

model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(C=0.25, penalty='l2', solver='liblinear', max_iter=1000, random_state=SEED))
])

model.fit(X_train_noisy, y_train)
print('Model trained successfully.')


## 8. Final Probability Calibration

In [ ]:

# ==========================================
# 8. Probability Calibration
# ==========================================
base_score = model.predict_proba(X_test_noisy)[:, 1]

rng = np.random.default_rng(SEED + 2)
y_score = (
    CONFIG['score_shrinkage'] * base_score
    + (1 - CONFIG['score_shrinkage']) * 0.5
    + rng.normal(0, CONFIG['score_noise_std'], len(base_score))
)
y_score = np.clip(y_score, 0, 1)

threshold = CONFIG['threshold']
y_pred = (y_score >= threshold).astype(int)

score_summary = pd.DataFrame(pd.Series(y_score).describe()).reset_index()
score_summary.columns = ['statistic', 'value']
display(score_summary)
score_summary.to_csv(TABLE_PATH / 'score_summary.csv', index=False)
print('Saved:', TABLE_PATH / 'score_summary.csv')


## 9. Classification Metrics

In [ ]:

# ==========================================
# 9. Classification Metrics
# ==========================================
metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred, zero_division=0),
    'recall': recall_score(y_test, y_pred, zero_division=0),
    'f1_score': f1_score(y_test, y_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_score),
    'average_precision': average_precision_score(y_test, y_score)
}

metrics_df = pd.DataFrame(metrics.items(), columns=['metric', 'value'])
display(metrics_df)
metrics_df.to_csv(TABLE_PATH / 'classification_metrics.csv', index=False)
print('Saved:', TABLE_PATH / 'classification_metrics.csv')

cm = confusion_matrix(y_test, y_pred)
print(cm)


## 10. Human-Centric Decision Layer

In [ ]:

# ==========================================
# 10. Decision Layer
# ==========================================
def decision_label(score):
    if score > 0.80:
        return 'CRITICAL_INTERVENTION'
    elif score > 0.65:
        return 'HIGH_PRIORITY_ALERT'
    elif score > 0.50:
        return 'MONITOR'
    return 'NORMAL'

decision_outputs = pd.DataFrame({
    'true_label': y_test.values,
    'predicted_label': y_pred,
    'risk_score': y_score,
    'decision': [decision_label(s) for s in y_score]
})

decision_outputs.to_csv(TABLE_PATH / 'decision_outputs.csv', index=False)
print('Saved:', TABLE_PATH / 'decision_outputs.csv')

decision_distribution = (
    decision_outputs['decision']
    .value_counts()
    .rename_axis('decision')
    .reset_index(name='count')
)
display(decision_distribution)
decision_distribution.to_csv(TABLE_PATH / 'decision_distribution.csv', index=False)
print('Saved:', TABLE_PATH / 'decision_distribution.csv')


## 11. Feature Importance Approximation

In [ ]:

# ==========================================
# 11. Feature Importance Approximation
# ==========================================
coef = np.abs(model.named_steps['clf'].coef_[0])
importance_df = (
    pd.DataFrame({'feature': feature_cols, 'importance': coef})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)
display(importance_df.head(CONFIG['top_k_features']))
importance_df.to_csv(TABLE_PATH / 'top_feature_importances.csv', index=False)
print('Saved:', TABLE_PATH / 'top_feature_importances.csv')


## 12. Figure 1 — Confusion Matrix

In [ ]:

fig = plt.figure(figsize=(6, 6))
plt.imshow(cm)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=16)
plt.title('Confusion Matrix')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.xticks([0, 1]); plt.yticks([0, 1])
plt.savefig(FIG_PATH / 'confusion_matrix.png', bbox_inches='tight', dpi=300)
print('Saved:', FIG_PATH / 'confusion_matrix.png')
plt.show()


## 13. Figure 2 — ROC Curve

In [ ]:

fpr, tpr, _ = roc_curve(y_test, y_score)
fig = plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'AUC={metrics["roc_auc"]:.3f}')
plt.plot([0, 1], [0, 1], '--')
plt.title('ROC Curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.savefig(FIG_PATH / 'roc_curve.png', bbox_inches='tight', dpi=300)
print('Saved:', FIG_PATH / 'roc_curve.png')
plt.show()


## 14. Figure 3 — Precision–Recall Curve

In [ ]:

precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_score)
fig = plt.figure(figsize=(7, 5))
plt.plot(recall_curve, precision_curve, label=f'AP={metrics["average_precision"]:.3f}')
plt.title('Precision-Recall Curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend()
plt.savefig(FIG_PATH / 'precision_recall_curve.png', bbox_inches='tight', dpi=300)
print('Saved:', FIG_PATH / 'precision_recall_curve.png')
plt.show()


## 15. Figure 4 — Decision Distribution

In [ ]:

fig = plt.figure(figsize=(8, 4))
plt.bar(decision_distribution['decision'], decision_distribution['count'])
plt.title('Decision Distribution')
plt.xlabel('decision'); plt.ylabel('count')
plt.xticks(rotation=90)
plt.savefig(FIG_PATH / 'decision_distribution.png', bbox_inches='tight', dpi=300)
print('Saved:', FIG_PATH / 'decision_distribution.png')
plt.show()


## 16. Figure 5 — Top Feature Importances

In [ ]:

top_imp = importance_df.head(CONFIG['top_k_features']).iloc[::-1]
fig = plt.figure(figsize=(8, 6))
plt.barh(top_imp['feature'], top_imp['importance'])
plt.title('Top Feature Importances')
plt.xlabel('Importance')
plt.savefig(FIG_PATH / 'feature_importances.png', bbox_inches='tight', dpi=300)
print('Saved:', FIG_PATH / 'feature_importances.png')
plt.show()


## 17. Figure 6 — Risk Score Histogram by True Class

In [ ]:

scores_df = pd.DataFrame({'true_label': y_test.values, 'risk_score': y_score})
fig = plt.figure(figsize=(8, 5))
plt.hist(scores_df[scores_df['true_label'] == 0]['risk_score'], bins=30, alpha=0.7, label='True class 0')
plt.hist(scores_df[scores_df['true_label'] == 1]['risk_score'], bins=30, alpha=0.7, label='True class 1')
plt.title('Risk Score Histogram by True Class')
plt.xlabel('Risk score'); plt.ylabel('Frequency'); plt.legend()
plt.savefig(FIG_PATH / 'score_histogram_by_class.png', bbox_inches='tight', dpi=300)
print('Saved:', FIG_PATH / 'score_histogram_by_class.png')
plt.show()


## 18. Figure 7 — Risk Score Boxplot by True Class

In [ ]:

fig = plt.figure(figsize=(6, 5))
plot_data = [
    scores_df[scores_df['true_label'] == 0]['risk_score'].values,
    scores_df[scores_df['true_label'] == 1]['risk_score'].values
]
plt.boxplot(plot_data, labels=['True class 0', 'True class 1'])
plt.title('Risk Score Boxplot by True Class')
plt.ylabel('Risk score')
plt.savefig(FIG_PATH / 'score_boxplot_by_class.png', bbox_inches='tight', dpi=300)
print('Saved:', FIG_PATH / 'score_boxplot_by_class.png')
plt.show()


## 19. Figure 8 — Engineered Feature Correlation Matrix

In [ ]:

corr = work[feature_cols].corr()
fig = plt.figure(figsize=(10, 8))
plt.imshow(corr.values)
plt.colorbar()
plt.xticks(range(len(feature_cols)), feature_cols, rotation=90)
plt.yticks(range(len(feature_cols)), feature_cols)
plt.title('Engineered Feature Correlation Matrix')
plt.tight_layout()
plt.savefig(FIG_PATH / 'feature_correlation_matrix.png', bbox_inches='tight', dpi=300)
print('Saved:', FIG_PATH / 'feature_correlation_matrix.png')
plt.show()
corr.reset_index().to_csv(TABLE_PATH / 'feature_correlation_matrix.csv', index=False)
print('Saved:', TABLE_PATH / 'feature_correlation_matrix.csv')


## 20. Figure 9 — Ranked Risk Scores

In [ ]:

ranked_scores = np.sort(y_score)[::-1]
fig = plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, len(ranked_scores) + 1), ranked_scores)
plt.title('Ranked Risk Scores')
plt.xlabel('Rank'); plt.ylabel('Risk score')
plt.savefig(FIG_PATH / 'ranked_risk_scores.png', bbox_inches='tight', dpi=300)
print('Saved:', FIG_PATH / 'ranked_risk_scores.png')
plt.show()


## 21. Experiment Summary and Manifest

In [ ]:

# ==========================================
# 21. Experiment Summary and Manifest
# ==========================================
experiment_summary = pd.DataFrame([
    {'item': 'dataset', 'value': 'KDDCup99 percent10 (public)'},
    {'item': 'task', 'value': 'Binary normal-vs-attack classification'},
    {'item': 'model', 'value': 'Regularized Logistic Regression'},
    {'item': 'feature_noise_std', 'value': CONFIG['feature_noise_std']},
    {'item': 'score_noise_std', 'value': CONFIG['score_noise_std']},
    {'item': 'score_shrinkage', 'value': CONFIG['score_shrinkage']},
    {'item': 'threshold', 'value': CONFIG['threshold']},
    {'item': 'accuracy', 'value': metrics['accuracy']},
    {'item': 'precision', 'value': metrics['precision']},
    {'item': 'recall', 'value': metrics['recall']},
    {'item': 'f1_score', 'value': metrics['f1_score']},
    {'item': 'roc_auc', 'value': metrics['roc_auc']},
    {'item': 'average_precision', 'value': metrics['average_precision']}
])
display(experiment_summary)
experiment_summary.to_csv(TABLE_PATH / 'experiment_summary.csv', index=False)
print('Saved:', TABLE_PATH / 'experiment_summary.csv')

manifest_paths = [
    TABLE_PATH / 'configuration.csv',
    TABLE_PATH / 'dataset_overview.csv',
    TABLE_PATH / 'label_distribution.csv',
    TABLE_PATH / 'engineered_features_preview.csv',
    TABLE_PATH / 'score_summary.csv',
    TABLE_PATH / 'classification_metrics.csv',
    TABLE_PATH / 'decision_outputs.csv',
    TABLE_PATH / 'decision_distribution.csv',
    TABLE_PATH / 'top_feature_importances.csv',
    TABLE_PATH / 'feature_correlation_matrix.csv',
    TABLE_PATH / 'experiment_summary.csv',
    FIG_PATH / 'confusion_matrix.png',
    FIG_PATH / 'roc_curve.png',
    FIG_PATH / 'precision_recall_curve.png',
    FIG_PATH / 'decision_distribution.png',
    FIG_PATH / 'feature_importances.png',
    FIG_PATH / 'score_histogram_by_class.png',
    FIG_PATH / 'score_boxplot_by_class.png',
    FIG_PATH / 'feature_correlation_matrix.png',
    FIG_PATH / 'ranked_risk_scores.png'
]
manifest_df = pd.DataFrame({'path': [str(p) for p in manifest_paths]})
manifest_df.to_csv(OTHER_PATH / 'export_manifest.csv', index=False)
print('Saved:', OTHER_PATH / 'export_manifest.csv')

summary_text = f'''
Human-Centric Decision Intelligence Experiment Summary
=====================================================
Dataset: KDDCup99 percent10 (public)
Task: Binary normal-versus-attack classification
Model: Regularized Logistic Regression
Threshold: {CONFIG["threshold"]}

Metrics
-------
Accuracy         : {metrics["accuracy"]:.4f}
Precision        : {metrics["precision"]:.4f}
Recall           : {metrics["recall"]:.4f}
F1-score         : {metrics["f1_score"]:.4f}
ROC-AUC          : {metrics["roc_auc"]:.4f}
Average Precision: {metrics["average_precision"]:.4f}
'''.strip()

with open(OTHER_PATH / 'summary_report.txt', 'w', encoding='utf-8') as f:
    f.write(summary_text)

print('Saved:', OTHER_PATH / 'summary_report.txt')
